<a target="_blank" href="https://colab.research.google.com/github/wilhelm-lab/koina/blob/main/clients/python/test/notebooks/koinapy.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Koina Workshop

This notebook is prepared to be run in Google [Colaboratory](https://colab.research.google.com/).
It contains tasks that are designed to guide new users through the following topics:

1. How to install koina & load packages
2. Basic principle of how to get predictions via koina (explained based on fragment ion intensity prediction)
3. How to visualize experimental and/or predicted fragment ion intensity spectra
4. How to compare fragment ion intensity spectra predictions of different models via koina
5. Use Case Example: Validate identification of experimental HLA peptides
5. How to compare RT predictions via koina

# 1: Installation
Before using Koina, the package and dependencies need to be installed. This step is only required once on your notebook, but it may need to be repeated upon reloading Google Colab.

## Task 1.1

What are the requirements for Koina and where do you find this information? (Hint: check out the [README](https://github.com/wilhelm-lab/koina/blob/main/clients/python/README.md) in the github repository).

## Task 1.2

Execute the below code cells, which installs spectrum_utils and Koina.

In [ ]:
!python --version

In [ ]:
# os.system install required to prevent version conflicts on google colab
import os
os.system("pip install --quiet git+https://github.com/bittremieux/spectrum_utils seaborn koinapy")

## Task 1.3

For this notebook to work, a few packages need to be imported that provide the functions used in the following. They should already be installed as dependencies of Koins. Should you get an error here, check that installation of the required packages was successful.

Import the below packages and functions by executing the code in the cell.

In [ ]:
# Basic imports
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)

# To plot masss spectra
import spectrum_utils.plot as sup
import spectrum_utils.spectrum as sus

# To download fasta
import requests
import gzip
import subprocess

# To digest fasta
from pyteomics.fasta import FASTA
from pyteomics.parser import cleave

# Predictions with Koina
from koinapy import Koina

If this works, you have installed Koina correctly.

# 2: Get predictions via Koina

Koina provides predictions as a service. You can choose from a growing set of pretrained models, covering different peptide properties.

## Task 2.1

Where do you find information about the available models and how to access them? (Hint: on the [koina website](https://koina.proteomicsdb.org/docs))

## Task 2.2

Define below variables accordingly.

In [ ]:
server = "koina.wilhelmlab.org"   # the server to use for predictions
ssl = True   # whether to use a secure server connection using SSL (Secure Sockets Layer)

## Task 2.3

Run the next cell to instantiate an object storing all the information about the model & the server, required for the upcoming prediction.

In [ ]:
prosit2019 = Koina("Prosit_2019_intensity", server_url=server, ssl=ssl)

## Task 2.4

Run the next cell to create a toy dataset, a table with 3 example peptide sequences.

In [ ]:
data = pd.DataFrame(
    {
        "peptide_sequences": np.array(
            ["VLHPLEGAVVIIFK", "VLHPLEGAVVLIFK", "SGVSRKPAPG"]
        ),
        "precursor_charges": np.array([2, 2, 2]),
        "collision_energies": np.array([25, 25, 25]),
        "instrument_types": np.array(["LUMOS", "LUMOS", "LUMOS"]),
    }
)
# Checkout input data
data

## Task 2.5

Retrieve predictions for the toy dataset using the model & server defined above.

In [ ]:
# Get predictions
pred_prosit2019 = prosit2019.predict(data)

## Task 2.6

Explore the structure of the retrieved predictions. How is it build?

In [ ]:
print(pred_prosit2019.shape)
pred_prosit2019

# 3: Visualize fragment ion spectra using spectrum_utils

spectrum_utils is a Python package for efficient mass spectrometry data processing and visualization.

## Task 3.1

Where do you dinf information about spectrum_utils? (Hint: check the [github repository](https://github.com/bittremieuxlab/spectrum_utils) and the [documentation](https://spectrum-utils.readthedocs.io/en/latest/quickstart.html))

## Task 3.2

Initialize a spectrum_utils.spectrum (sus) MS2 spectrum instance with our predictions from above. Here, we only use the first peptide/first MS2 spectrum prediction.
Then annotate the MS2 spectrum instance with its peptide string in [ProForma](https://www.psidev.info/proforma) notation.
spectrum_utils.spectrum also contains a function for plotting these MS2 spectrum instances.

In [ ]:
# Initialize MS2 spectrum instance with the first predicted spectrum
bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_prosit2019["mz"][0], intensity=pred_prosit2019["intensities"][0]
)
# Annotate the spectrum with its ProForma string
bot_spectrum = bot_spectrum.annotate_proforma(
    "VLHPLEGAVVIIFK", 20, "ppm", ion_types="by", max_ion_charge=2
)

# Plot the spectrum
fig, ax = plt.subplots(figsize=(8, 4.5))
sup.spectrum(bot_spectrum, ax=ax)

## Task 3.3

Initialize a spectrum_utils.spectrum (sus) MS2 spectrum instance with an Universal Spectrum Identifier (USI). A USI points directly to a specific mass spectrum inside a public data repository.

What is different between this spectrum plot and the one from Task 3.2?

In [ ]:
# Initialize a sus.spectrum with a published experimental spectrum
top_spectrum = sus.MsmsSpectrum.from_usi(
    "mzspec:PXD000561:Adult_Frontalcortex_bRP_Elite_85_f09:scan:17555:VLHPLEGAVVIIFK/2"
)
# Annotate the spectrum with its ProForma string.
top_spectrum = top_spectrum.annotate_proforma(
    "VLHPLEGAVVIIFK", 20, "ppm", ion_types="by", max_ion_charge=2
)

# Plot the spectrum.
fig, ax = plt.subplots(figsize=(8, 4.5))
sup.spectrum(top_spectrum, ax=ax)

## Task 3.4

Create a mirror plot for easy comparison of the two spectra above.
spectrum_utils.spectrum also contains a function for creating mirror plots of two spectra.

In [ ]:
# Create mirror plot of the predicted and experimental spectrum
fig, ax = plt.subplots(figsize=(8, 4.5))
sup.mirror(top_spectrum, bot_spectrum, ax=ax)

# 4: Compare predictions of different models

For most peptide properties, Koina provides several different pretrained models. The next topic introduces how to compare predictions of different models.

## Task 4.1

Instantiate an object for each of the three following fragment ion intensity prediction models:

* [Prosit](https://koina.proteomicsdb.org/docs#post-/Prosit_2020_intensity_HCD/infer)
* [MS2PIP](https://koina.proteomicsdb.org/docs#post-/ms2pip_HCD2021/infer)
* [AlphaPeptDeep](https://koina.proteomicsdb.org/docs#post-/AlphaPeptDeep_ms2_generic/infer)

Then create predictions.

In [ ]:
prosit2020 = Koina("Prosit_2020_intensity_HCD", server_url=server, ssl=ssl)
ms2pip = Koina("ms2pip_HCD2021", server_url=server, ssl=ssl)
alphapep = Koina("AlphaPept_ms2_generic", server_url=server, ssl=ssl)

# create predictions for the toy data created above using the three different models
pred_prosit2020 = prosit2020.predict(data)
pred_ms2pip = ms2pip.predict(data)
pred_alphapep = alphapep.predict(data)

## Task 4.2

Create mirror plots for all 3 predictions, each in combination with the experimental spectrum from Task 3.3.

In [ ]:
# Prosit prediction vs published experimental spectrum

bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_prosit2020["mz"][0], intensity=pred_prosit2020["intensities"][0]
)
bot_spectrum = bot_spectrum.annotate_proforma(
    "VLHPLEGAVVIIFK", 20, "ppm", ion_types="by", max_ion_charge=2
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sup.mirror(top_spectrum, bot_spectrum, ax=ax)
ax.set_title("Prosit_2020_intensity_HCD vs. published experimental spectrum")

In [ ]:
# ms2pip prediction vs published experimental spectrum

bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_ms2pip["mz"][0], intensity=pred_ms2pip["intensities"][0]
)
bot_spectrum = bot_spectrum.annotate_proforma(
    "VLHPLEGAVVIIFK", 20, "ppm", ion_types="by", max_ion_charge=2
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sup.mirror(top_spectrum, bot_spectrum, ax=ax)
ax.set_title("ms2pip_HCD2021 vs. published experimental spectrum")

In [ ]:
# AlphaPept prediction vs published experimental spectrum
bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_alphapep["mz"][0], intensity=pred_alphapep["intensities"][0]
)
bot_spectrum = bot_spectrum.annotate_proforma(
    "VLHPLEGAVVIIFK", 20, "ppm", ion_types="by", max_ion_charge=2
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sup.mirror(top_spectrum, bot_spectrum, ax=ax)
ax.set_title("AlphaPept_ms2_generic vs. published experimental spectrum")

# 5: Use Case Example: Validate identification of experimental HLA peptide spectrum

For this topic, an experimental spectrum, that was initially identified with the peptide sequence "SGVSRKPAPG" and was used as evidence for a novel HLA peptide, is compared to the spectrum of a synthetic peptide with the same sequence "SGVSRKPAPG". [([Mylonas et al. 2018], Figure 2A)](https://www.mcponline.org/article/S1535-9476(20)31999-X/fulltext)

Both experimental spectra are then compared to predicted spectra for that peptide sequence.

## Task 5.1

Compare the experimental spectrum, initially identified with the peptide sequence "SGVSRKPAPG" with the spectrum of a synthetic peptide with the same sequence "SGVSRKPAPG".

What do you observe? Do the spectra agree?

In [ ]:
# initial identification of the experimental spectrum
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD000394:20130504_EXQ3_MiBa_SA_Fib-2:scan:4234")
top_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# Spectrum of a synthetic peptide with the same sequence SGVSRKPAPG
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD010793:20170817_QEh1_LC1_HuPa_SplicingPep_10pmol_G2_R01:scan:8296")
bot_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.set_title("Experimental spectrum initially identified as SGVSRKPAPG vs.\nspectrum of a synthetic peptide with sequence SGVSRKPAPG")
sup.mirror(top_spectrum, bot_spectrum, ax=ax)

## Task 5.2

Both experimental spectra are now compared to the predicted spectra for the peptide sequence "SGVSRKPAPG", created in Task 4.1.

Predictions were created from these models:

* [Prosit](https://koina.proteomicsdb.org/docs#post-/Prosit_2020_intensity_HCD/infer)
* [MS2PIP](https://koina.proteomicsdb.org/docs#post-/ms2pip_HCD2021/infer)
* [AlphaPeptDeep](https://koina.proteomicsdb.org/docs#post-/AlphaPeptDeep_ms2_generic/infer)

What do you observe? Which spectra agree?

In [ ]:
# for Prosit

# initial identification of the experimental spectrum
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD000394:20130504_EXQ3_MiBa_SA_Fib-2:scan:4234")
top_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# Spectrum of a synthetic peptide with the same sequence SGVSRKPAPG
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD010793:20170817_QEh1_LC1_HuPa_SplicingPep_10pmol_G2_R01:scan:8296")
top2_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# Prosit_2019_intensity prediction
bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_prosit2019["mz"][2], intensity=pred_prosit2019["intensities"][2]
)
bot_spectrum = bot_spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby", max_ion_charge=2
)


# 2 mirror plots 
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

axes[0].set_title(
    "Experimental spectrum initially identified as SGVSRKPAPG vs.\nProsit_2019_intensity prediction"
)
sup.mirror(top_spectrum, bot_spectrum, ax=axes[0])

axes[1].set_title("Spectrum of a synthetic peptide with sequence SGVSRKPAPG vs.\nProsit_2019_intensity prediction")
sup.mirror(top2_spectrum, bot_spectrum, ax=axes[1])

plt.tight_layout()

In [ ]:
# for MS2PIP

# initial identification of the experimental spectrum
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD000394:20130504_EXQ3_MiBa_SA_Fib-2:scan:4234")
top_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# Spectrum of a synthetic peptide with the same sequence SGVSRKPAPG
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD010793:20170817_QEh1_LC1_HuPa_SplicingPep_10pmol_G2_R01:scan:8296")
top2_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# ms2pip_HCD2021 prediction
bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_ms2pip["mz"][2], intensity=pred_ms2pip["intensities"][2]
)
bot_spectrum = bot_spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby", max_ion_charge=2
)


# 2 mirror plots 
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

axes[0].set_title(
    "Experimental spectrum initially identified as SGVSRKPAPG vs.\nms2pip_HCD2021 prediction"
)
sup.mirror(top_spectrum, bot_spectrum, ax=axes[0])

axes[1].set_title("Spectrum of a synthetic peptide with sequence SGVSRKPAPG vs.\nms2pip_HCD2021 prediction")
sup.mirror(top2_spectrum, bot_spectrum, ax=axes[1])

plt.tight_layout()

In [ ]:
# for AlphaPeptDeep

# initial identification of the experimental spectrum
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD000394:20130504_EXQ3_MiBa_SA_Fib-2:scan:4234")
top_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# Spectrum of a synthetic peptide with the same sequence SGVSRKPAPG
spectrum = sus.MsmsSpectrum.from_usi("mzspec:PXD010793:20170817_QEh1_LC1_HuPa_SplicingPep_10pmol_G2_R01:scan:8296")
top2_spectrum = spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby",max_ion_charge=2
)

# AlphaPept_ms2_generic prediction
bot_spectrum = sus.MsmsSpectrum(
    "0", 0, 2, mz=pred_alphapep["mz"][2], intensity=pred_alphapep["intensities"][2]
)
bot_spectrum = bot_spectrum.annotate_proforma(
    "SGVSRKPAPG", 10, "ppm", ion_types="aby", max_ion_charge=2
)


# 2 mirror plots 
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

axes[0].set_title(
    "Experimental spectrum initially identified as SGVSRKPAPG vs.\nAlphaPept_ms2_generic prediction"
)
sup.mirror(top_spectrum, bot_spectrum, ax=axes[0])

axes[1].set_title("Spectrum of a synthetic peptide with sequence SGVSRKPAPG vs.\nAlphaPept_ms2_generic prediction")
sup.mirror(top2_spectrum, bot_spectrum, ax=axes[1])

plt.tight_layout()

# 6: Retention time comparison

Next to fragment ion intensity prediction, indexed retention time (iRT) prediction plays an important role.

## Task 6.1 

Download and unpack the example reference proteome we are going to use for the next topic.

In [ ]:
url = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/reference_proteomes/Eukaryota/UP000005640/UP000005640_9606.fasta.gz"
myfile = requests.get(url)

with open("UP000005640_9606.fasta.gz", "wb") as f_in:
    f_in.write(myfile.content)

_ = subprocess.run(["gunzip", "UP000005640_9606.fasta.gz"])

## Task 6.2

In silico digest the protein sequences in the fasta file to peptides and create a table.

In [ ]:
# Digest fasta
peptides = set()
for prot in FASTA("UP000005640_9606.fasta"):
    peptides.update(cleave(prot.sequence, "trypsin", min_length=7, max_length=30))

# Create dataframe with peptides
data = pd.DataFrame({"peptide_sequences": np.array(list(peptides))})

# filter out peptides with unknown amino acids and replace C with C[UNIMOD:4] for carbamidomethylation
data = data[~data["peptide_sequences"].str.contains("X|U")]
data["peptide_sequences"] = data["peptide_sequences"].str.replace("C", "C[UNIMOD:4]")

data.head(20)

## Task 6.3

Retrieve iRT predictions from the [Prosit](https://koina.proteomicsdb.org/docs#post-/Prosit_2019_irt/infer) and the [DeepLC](https://koina.proteomicsdb.org/docs#post-/Deeplc_hela_hf/infer) model via Koina. The predictions are added to the table with peptide sequences.

In [ ]:
prosit2019_irt = Koina("Prosit_2019_irt", server_url=server, ssl=ssl)
deeplc = Koina("Deeplc_hela_hf", server_url=server, ssl=ssl)

data["prosit_irt"] = prosit2019_irt.predict(data)["irt"]
data["deeplc_irt"] = deeplc.predict(data)["irt"]

## Task 6.4 

Plot the distributions of predicted iRTs for each model.

In [ ]:
# Prosit iRT predictions
sns.histplot(data, x="prosit_irt")

In [ ]:
# DeepLC iRT predictions
sns.histplot(data, x="deeplc_irt")

## Task 6.5

Compare both iRT predictions.

In [ ]:
sns.histplot(data, x="prosit_irt", y="deeplc_irt")
plt.title("Prosit_2019_irt vs. Deeplc_hela_hf iRT predictions\n Pearson correlation: " + str(round(data[["prosit_irt", "deeplc_irt"]].corr(method="pearson").iloc[0,1], 3)))